# Load Dependencies

In [1]:
# Data analysis libraries
import numpy as np
import pandas as pd; pd.options.display.max_columns = 200
import geopandas as gpd
import linref as lr

# Visualization libraries
import plotly.express as px

In [2]:
# Define global variables
PROJECT_CRS = 'EPSG:3857'

# Load Data

In [3]:
# Point this to the location of the Geopackage file
fp = '/mnt/h/H0000/00CMH.00251.00_S_ODOT HSIP Safety Services/02C Python Training/Example Data/franklin_county_training_data.gpkg'

# List all the layers in the file
gpd.list_layers(fp)

,name,geometry_type
0,crashes,Point
1,roadways,MultiLineString
2,crashes_enriched,Point


In [14]:
# Load roadway data
roadways = gpd.read_file(fp, layer='roadways')
roadways.to_crs(PROJECT_CRS, inplace=True)

# Load crash data
query = """
SELECT OBJECTID, CRASH_YR, KABCO, CRASH_TYPE_SIMPLE, DAY_IN_WEEK_TEXT, HOUR_PERIOD, geom
FROM crashes_enriched
"""
crashes = gpd.read_file(fp, sql=query)
crashes.to_crs(PROJECT_CRS, inplace=True)

print(f'Data loaded: {len(roadways):,.0f} roadways, {len(crashes):,.0f} crashes')

Data loaded: 5,110 roadways, 25,872 crashes


# Crash Scoring Metrics

In [15]:
# Create a series of crash scoring metrics based on crash severity and mode
# These will be used for creating a variety of high-injury networks for each defined
# scoring metric

# We will first create boolean masks for each of the scoring metrics

# Crash severity metrics
mask_kabc = crashes['KABCO'].isin(['K', 'A', 'B', 'C'])
mask_ka   = crashes['KABCO'].isin(['K', 'A'])

# Crash mode metrics
mask_ped = crashes['CRASH_TYPE_SIMPLE'].isin(['Pedestrian'])
mask_pdc = crashes['CRASH_TYPE_SIMPLE'].isin(['Pedalcycle'])
mask_veh = ~(mask_ped | mask_pdc)

In [16]:
# Combined metrics
crashes['SCORE_KABC_VEH'] = mask_kabc * mask_veh * 1
crashes['SCORE_KABC_PED'] = mask_kabc * mask_ped * 1
crashes['SCORE_KABC_PDC'] = mask_kabc * mask_pdc * 1
crashes['SCORE_KA_VEH']   = mask_ka   * mask_veh * 1
crashes['SCORE_KA_PED']   = mask_ka   * mask_ped * 1
crashes['SCORE_KA_PDC']   = mask_ka   * mask_pdc * 1

In [17]:
# How can we create metrics for late night weekend crashes?
# E.g., Friday and Saturday nights between 9 PM and 3 AM

In [22]:
crashes['SCORE_KA_PED'].sum()

np.int64(413)

In [21]:
crashes.head()

,OBJECTID,CRASH_YR,KABCO,CRASH_TYPE_SIMPLE,DAY_IN_WEEK_TEXT,HOUR_PERIOD,geometry,SCORE_KABC_VEH,SCORE_KABC_PED,SCORE_KABC_PDC,SCORE_KA_VEH,SCORE_KA_PED,SCORE_KA_PDC
0,192005196,2019,B,Angle,Tue,12PM-3PM,POINT (-9251476.789 4851166.664),1,0,0,0,0,0
1,192005229,2019,C,Rear End,Tue,6AM-9AM,POINT (-9248208.671 4851567.152),1,0,0,0,0,0
2,192006158,2019,C,Lane Departure,Wed,12PM-3PM,POINT (-9245611.142 4848270.574),1,0,0,0,0,0
3,192013210,2019,K,Lane Departure,Wed,9PM-12AM,POINT (-9227900.656 4878031.178),1,0,0,1,0,0
4,192017264,2019,K,Pedalcycles,Sun,6PM-9PM,POINT (-9227842.882 4878650.276),1,0,0,1,0,0


# Linear Referencing

In [ ]:
# Create events collections for managing linearly referenced data
roadways_ec = lr.EventsCollection(roadways, keys=['NLF_ID'], beg='CTL_BEGIN_', end='CTL_END_NB', geom='geometry')
crashes_ec = lr.EventsCollection(crashes, keys=['NLFID'], beg='COUNTY_LOG_NBR', geom='geometry')

In [ ]:
get_columns = ['SPEED_LIMI', 'LANES_NBR', 'LANE_WIDTH']
crashes_ec.df[get_columns] = crashes_ec.merge(roadways_ec)[get_columns].most()

# Crash Distribution Analysis

_Linref-based Crash Distribution Schematic Example_

<img src="../99_Resources/img/linref-distribution-schematic.png" width=1000>